# Week 3 — Expand to 200+ images, build the training Dataset

Doubles the Week 1 dataset with more synthetic variety (different fonts and
render styles are cheap; the handout is right that variety is what makes a
model robust) and wraps everything in `src/dataset.py`'s `UrduOCRDataset`,
which — vs. the handout's version — normalises Urdu text, preprocesses
images, and applies train-time augmentation (see that file's docstring).


In [ ]:
# Run once per Colab session (safe to re-run: pulls latest instead of
# silently reusing a stale checkout from an earlier run this session).
import os
import subprocess
import sys

REPO_DIR = '/content/repo'
# IMPORTANT: pinned to this specific branch, not the repo's default branch
# -- see Quickstart_Colab.ipynb's Cell 1 for why (an unqualified clone/pull
# can silently track a branch that's missing later fixes, and `git pull`
# then reports "Already up to date" against the wrong branch with no
# warning). Replace with your own fork's URL/branch if running elsewhere,
# or with 'main' once this branch has actually been merged there.
BRANCH = 'claude/high-accuracy-solution-r9r2t1'
if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} --single-branch https://github.com/Rameesha-Malik/ML-SI-Solution-Urdu-OCR.git {REPO_DIR}
else:
    print(f'{REPO_DIR} already exists -- fetching + resetting to origin/{BRANCH} instead of re-cloning.')
    !git -C {REPO_DIR} fetch origin {BRANCH}
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} reset --hard origin/{BRANCH}

os.chdir(REPO_DIR)
!git -C {REPO_DIR} log -1 --format='Repo HEAD: %h %s'

# Deliberately NOT --upgrade-strategy eager, and NOT --force-reinstall on
# the whole file: both would make pip consider reinstalling `torch` even
# though Colab's pre-installed torch already satisfies our ">=2.2.0" bound
# -- and a freshly pip-resolved torch build was observed, in testing, to be
# missing a CUDA sub-package it needs (ImportError: libcublas.so.13), which
# would trade a fixable Python-level bug for a broken GPU runtime. Plain
# `--upgrade -r requirements.txt` still fully resolves and changes
# transformers/pillow/gradio (their pins are strict enough that pip must
# change them regardless of strategy -- upgrade-strategy only affects
# whether *unnamed transitive* dependencies like torch get proactively
# touched), while leaving torch itself alone.
!pip install -q --upgrade -r requirements.txt

# Ground-truth check of what's on disk, via a subprocess (`pip show`) --
# NOT `import transformers`/`import PIL`. If a notebook cell earlier in
# this session already imported one of these, Python caches that module
# in memory for the rest of the process; reinstalling on disk does not
# change what's already loaded there. The actual training/eval scripts
# run as a separate `!python ...` subprocess each time, so they always
# see what's genuinely on disk -- this mirrors that rather than this
# kernel's own (possibly stale) view.
def _installed_version(pkg):
    result = subprocess.run(['pip', 'show', pkg], capture_output=True, text=True)
    return next(
        (l.split(':', 1)[1].strip() for l in result.stdout.splitlines() if l.startswith('Version:')),
        '?',
    )

versions = {pkg: _installed_version(pkg) for pkg in ('transformers', 'pillow', 'gradio')}
print('Installed on disk:', versions)

if 'src' not in sys.path:
    sys.path.insert(0, 'src')

expected_major = {'transformers': ('4.',), 'pillow': ('11.',), 'gradio': ('4.', '5.', '6.0', '6.1')}
bad = {
    p: v for p, v in versions.items()
    if not any(v.startswith(prefix) for prefix in expected_major[p])
}
if bad:
    raise RuntimeError(
        f'Unexpected version(s) on disk: {bad} (expected transformers 4.x, '
        'pillow 11.x, gradio <6.20 per requirements.txt). pip could not '
        'fully satisfy the pins here -- likely another package in this '
        'Colab image forces a conflicting version. Re-run this cell and '
        'share the FULL pip output above (not just this message) so the '
        'real conflicting package can be identified.'
    )

# Functional smoke test, not just a version check -- and run in a genuinely
# fresh subprocess, not this notebook kernel. Pillow (and any package with a
# compiled C extension) cannot have its native .so swapped out in an
# already-running process even after a correct on-disk reinstall; the only
# way to know it will actually work is to check from a brand new process,
# which is also exactly what the training/eval scripts do.
pil_check = subprocess.run(
    ['python3', '-c',
     "from PIL import Image; im = Image.new('RGB', (32, 32), 'white'); "
     "im.save('/tmp/_pil_smoke_test.png'); "
     "Image.open('/tmp/_pil_smoke_test.png').load(); print('PIL round-trip OK')"],
    capture_output=True, text=True,
)
print(pil_check.stdout.strip())
if pil_check.returncode != 0:
    raise RuntimeError(
        'Pillow is installed but broken (fails to save/load an image even in '
        'a fresh process) -- this is not a version-pin issue, it is a '
        'corrupted install, most likely leftover mixed files on disk from '
        'earlier pip installs/reinstalls in this same Colab VM. '
        '`Runtime > Restart session` will NOT fix this (it only restarts '
        'the Python process, not the disk). You need '
        '`Runtime > Disconnect and delete runtime`, then reconnect to a '
        'brand new VM and run this notebook from Cell 1 on that clean '
        'slate.\n\nOriginal error:\n' + pil_check.stderr[-1500:]
    )

print('Setup complete -- all pinned versions correctly installed on disk and working.')


## Step 1 — Expand to 200+ images

In [ ]:
from synthetic import generate_dataset, merge_labels_csv

# Generate more synthetic variety on top of Week 1's set (different random seed).
stats = generate_dataset(
    output_dir='data/raw/synthetic',
    labels_csv='data/labels_synthetic_2.csv',
    num_images=150,
    pseudo_fraction=0.3,
    seed=123,
    image_prefix='synth_w3',
)
print(f'Added {stats.total} more images')

# Re-merge everything (synthetic batch 1 + batch 2 + your real photos) into labels.csv.
n = merge_labels_csv(
    ['data/labels_synthetic.csv', 'data/labels_synthetic_2.csv', 'data/labels_real.csv'],
    'data/labels.csv',
)
print(f'data/labels.csv now has {n} total labelled images')
print('Keep adding real photographed images to data/raw/ + data/labels_real.csv — '
      'real data matters more than more synthetic data past this point. See '
      'docs/ACCURACY_NOTES.md.')


Then preprocess anything newly added (skips files already in `data/processed/`):

In [ ]:
from preprocessing import preprocess_folder
count = preprocess_folder('data/raw', 'data/processed')
print(f'{count} images in data/processed/')


## Step 2 — Build the train/test split + Dataset class

`split_dataset` writes explicit `data/train.csv` / `data/test.csv` files (reproducible, diffable) rather than an in-memory `random_split` — Week 4's training script reads these directly.

In [ ]:
from dataset import split_dataset, UrduOCRDataset
from transformers import TrOCRProcessor

n_train, n_test = split_dataset(
    'data/labels.csv', 'data/train.csv', 'data/test.csv', test_size=0.15, seed=42,
)
print(f'Training samples: {n_train}')
print(f'Testing samples:  {n_test}')

processor = TrOCRProcessor.from_pretrained('microsoft/trocr-base-printed')
train_dataset = UrduOCRDataset('data/train.csv', processor, train=True)
test_dataset = UrduOCRDataset('data/test.csv', processor, train=False)

sample = train_dataset[0]
print('Sample pixel_values shape:', sample['pixel_values'].shape)
print('Sample labels shape:', sample['labels'].shape)
print('Dataset is working correctly!')
